# Эксперимент 1: квадратичная цель на L1-шаре

Сравнение классического Frank–Wolfe и **Away-step Frank–Wolfe** на задаче
$\min_{\|x\|_1 \le R} \tfrac12 x^\top A x - b^\top x$ с Armijo вдоль допустимого отрезка.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents][:10]:
        if (p / "src" / "optimization.py").is_file():
            return p
    return here


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.oracles import QuadraticOracle
from src.optimization import away_step_frank_wolfe_method, frank_wolfe_method
from src.paths import figs_dir

FIG = figs_dir()
np.random.seed(0)

In [ ]:
n = 40
rng = np.random.default_rng(1)
A = rng.standard_normal((n, n))
A = A.T @ A + 0.5 * np.eye(n)
b = rng.standard_normal(n)
oracle = QuadraticOracle(A, b)

x0 = np.zeros(n)
R = 3.0
tol = 5e-3
max_iter = 8000

x_fw, msg_fw, h_fw = frank_wolfe_method(
    oracle, x0, R, tolerance=tol, max_iter=max_iter, step_size_strategy="armijo", trace=True
)
x_afw, msg_afw, h_afw = away_step_frank_wolfe_method(
    oracle, x0, R, tolerance=tol, max_iter=max_iter, step_size_strategy="armijo", trace=True
)

print("FW:", msg_fw, "f* ≈", oracle.func(x_fw), "итераций", len(h_fw["func"]))
print("AFW:", msg_afw, "f* ≈", oracle.func(x_afw), "итераций", len(h_afw["func"]))

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(h_fw["fw_gap"], label="FW gap (Frank–Wolfe)")
plt.semilogy(h_afw["fw_gap"], label="FW gap (Away-step FW)")
plt.xlabel("Итерация")
plt.ylabel("Frank–Wolfe gap")
plt.title("Квадратичная задача: сходимость по gap")
plt.legend()
plt.grid(True, which="both", ls=":", alpha=0.6)
plt.tight_layout()
out = FIG / "exp01_quadratic_fw_gap.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print("Сохранено:", out)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(h_fw["func"], label="FW: f(x_k)")
plt.plot(h_afw["func"], label="AFW: f(x_k)")
plt.xlabel("Итерация")
plt.ylabel("Значение f")
plt.title("Квадратичная задача: убывание целевой функции")
plt.legend()
plt.grid(True, ls=":", alpha=0.6)
plt.tight_layout()
out = FIG / "exp01_quadratic_objective.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print("Сохранено:", out)